# Task 2: Fine-tuning de un modelo preentrenado (Keras)

1. Configuración e imports
2. Split train/val/test (se crea una vez y se guarda en `split_seed42.json`)
3. Datasets y modelo
4. Entrenamiento y evaluación (Route A: VGG16 sobre espectrogramas)
5. Otros backbones / experimentos


In [1]:
import os, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# ---------- CONFIGURACIÓN ----------
SEED = 42
keras.utils.set_random_seed(SEED)

DATA_DIR   = Path("/Users/annacasanovaspoirier/Desktop/MSC/Machine Learning for Sound and Music/music-genre-classification-2/data/images_original")
WAV_DIR    = DATA_DIR.parent / "genres_original"      # solo para VGGish (Route B)
SPLIT_FILE = Path("split_seed42.json")                # se guarda junto al notebook

IMG_SIZE = (224, 224)      # si va muy lento en CPU prueba (160, 160)
BATCH    = 32
WEIGHTS  = "imagenet"      # None = sin preentrenar (solo para tests)
# -----------------------------------

assert DATA_DIR.exists(), f"No existe {DATA_DIR}"
print("TF", tf.__version__, "| Keras", keras.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

TF 2.21.0 | Keras 3.15.1
GPUs: []


## 1. Split train / val / test

Se crea **una sola vez** y se guarda. Si `split_seed42.json` ya existe, se carga (así Task 1, Task 2 y Lab 1 pueden usar exactamente las mismas canciones).

In [2]:
# Géneros = subcarpetas (se ignoran archivos ocultos como .DS_Store)
genres = sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir() and not p.name.startswith("."))
class_to_idx = {g: i for i, g in enumerate(genres)}
N_CLASSES = len(genres)
print(genres, N_CLASSES)
assert N_CLASSES == 10, "Debería haber 10 géneros"

def rel(p):
    """Ruta relativa 'genero/archivo.png' (sirve también si el JSON antiguo tenía rutas absolutas)."""
    p = Path(p)
    return f"{p.parent.name}/{p.name}"

def to_labels(rels):
    return np.array([class_to_idx[r.split("/")[0]] for r in rels], dtype=np.int32)

if SPLIT_FILE.exists():
    split = json.load(open(SPLIT_FILE))
    tr_r, va_r, te_r = [np.array([rel(p) for p in split[k]]) for k in ("train", "val", "test")]
    print(f"Split cargado de {SPLIT_FILE}")
else:
    all_r = np.array(sorted(rel(p) for p in DATA_DIR.glob("*/*.png")))
    all_y = to_labels(all_r)
    tr_r, te_r = train_test_split(all_r, test_size=0.2, stratify=all_y, random_state=SEED)
    tr_r, va_r = train_test_split(tr_r, test_size=0.1, stratify=to_labels(tr_r), random_state=SEED)
    json.dump({"train": tr_r.tolist(), "val": va_r.tolist(), "test": te_r.tolist()},
              open(SPLIT_FILE, "w"), indent=1)
    print(f"Split nuevo guardado en {SPLIT_FILE}")

# Rutas completas y etiquetas (enteros 0..9)
tr_p = np.array([str(DATA_DIR / r) for r in tr_r]); tr_y = to_labels(tr_r)
va_p = np.array([str(DATA_DIR / r) for r in va_r]); va_y = to_labels(va_r)
te_p = np.array([str(DATA_DIR / r) for r in te_r]); te_y = to_labels(te_r)

# Comprobaciones
assert not (set(tr_r) & set(te_r)) and not (set(tr_r) & set(va_r)) and not (set(va_r) & set(te_r)), "¡Solapamiento!"
assert all(os.path.exists(p) for p in np.concatenate([tr_p, va_p, te_p])), "Faltan archivos"
print("train/val/test:", len(tr_p), len(va_p), len(te_p))
print("por género  train:", np.bincount(tr_y), "\n            val:  ", np.bincount(va_y), "\n            test: ", np.bincount(te_y))

['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock'] 10
Split cargado de split_seed42.json
train/val/test: 719 80 200
por género  train: [72 72 72 72 72 71 72 72 72 72] 
            val:   [8 8 8 8 8 8 8 8 8 8] 
            test:  [20 20 20 20 20 20 20 20 20 20]


## 2. Datasets y modelo

In [3]:
BACKBONES = {
    "vgg16":    (keras.applications.VGG16,          keras.applications.vgg16.preprocess_input),
    "resnet50": (keras.applications.ResNet50,       keras.applications.resnet50.preprocess_input),
    "effnetb0": (keras.applications.EfficientNetB0, keras.applications.efficientnet.preprocess_input),
}

def make_ds(paths, labels, preprocess, train=False):
    paths  = np.asarray([str(p) for p in paths])
    labels = np.asarray(labels, dtype=np.int32)
    def load(p, y):
        img = tf.io.read_file(p)
        img = tf.image.decode_png(img, channels=3)          # descarta el canal alpha
        img = tf.image.resize(img, IMG_SIZE)
        return preprocess(img), y
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if train:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

def build_model(name, dropout=0.3, dense_units=0):
    Base, _ = BACKBONES[name]
    base = Base(include_top=False, weights=WEIGHTS, input_shape=IMG_SIZE + (3,))
    base.trainable = False
    inp = keras.Input(IMG_SIZE + (3,))
    x = base(inp, training=False)                 # BatchNorm en modo inferencia
    x = layers.GlobalAveragePooling2D()(x)
    if dense_units:
        x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(N_CLASSES, activation="softmax")(x)
    return keras.Model(inp, out), base

def new_callbacks():
    return [keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5,
                                          restore_best_weights=True)]

def compile_model(model, lr):
    # OJO: argumentos con nombre (en Keras 3 el orden posicional cambió)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

def run_experiment(name, head_epochs=15, ft_epochs=20, unfreeze_last=30,
                   lr_head=1e-3, lr_ft=1e-5, dropout=0.3, dense_units=0):
    """Fase 1: entrena solo la cabeza. Fase 2: descongela las últimas `unfreeze_last` capas.
    Usa val para decidir; el test solo se evalúa al final."""
    _, prep = BACKBONES[name]
    train_ds = make_ds(tr_p, tr_y, prep, train=True)
    val_ds   = make_ds(va_p, va_y, prep)
    test_ds  = make_ds(te_p, te_y, prep)

    model, base = build_model(name, dropout, dense_units)

    t0 = time.time()
    # Fase 1: solo la cabeza
    compile_model(model, lr_head)
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=head_epochs,
                   callbacks=new_callbacks(), verbose=2)

    # Fase 2: fine-tuning de las últimas capas con lr bajo
    h2 = None
    if unfreeze_last > 0 and ft_epochs > 0:
        base.trainable = True
        for l in base.layers[:-unfreeze_last]:
            l.trainable = False
        for l in base.layers:                      # BatchNorm siempre congelada
            if isinstance(l, layers.BatchNormalization):
                l.trainable = False
        compile_model(model, lr_ft)                # recompilar tras cambiar trainable
        h2 = model.fit(train_ds, validation_data=val_ds, epochs=ft_epochs,
                       callbacks=new_callbacks(), verbose=2)
    train_time = time.time() - t0

    val_acc = model.evaluate(val_ds, verbose=0)[1]

    # Test (una sola vez, con la configuración ya elegida)
    t0 = time.time()
    probs = model.predict(test_ds, verbose=0)
    infer_time = (time.time() - t0) / len(te_p)
    preds = probs.argmax(1)
    acc = accuracy_score(te_y, preds)
    n_params = model.count_params()
    print(f"\n{name}: val acc={val_acc:.4f} | TEST acc={acc:.4f} | train {train_time:.0f}s | "
          f"{infer_time*1000:.1f} ms/canción | params {n_params:,}")
    return dict(name=name, model=model, probs=probs, preds=preds, acc=acc, val_acc=val_acc,
                train_time=train_time, infer_time=infer_time, params=n_params,
                hist=(h1.history, h2.history if h2 else {}))

def evaluate(res, title):
    print(classification_report(te_y, res["preds"], target_names=genres, digits=3))
    cm = confusion_matrix(te_y, res["preds"])
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=genres, yticklabels=genres, cmap="Blues")
    plt.title(f"{title} (acc={res['acc']:.3f})"); plt.ylabel("Real"); plt.xlabel("Predicho")
    plt.tight_layout(); plt.show()

def plot_history(res, title=""):
    h1, h2 = res["hist"]
    acc = h1["accuracy"] + h2.get("accuracy", []); vacc = h1["val_accuracy"] + h2.get("val_accuracy", [])
    plt.figure(figsize=(6, 4))
    plt.plot(acc, label="train"); plt.plot(vacc, label="val")
    plt.axvline(len(h1["accuracy"]) - 0.5, color="gray", ls="--", label="inicio fine-tuning")
    plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.title(title); plt.legend(); plt.show()

## 3. Comprobación rápida antes de entrenar

In [4]:
x, y = next(iter(make_ds(tr_p, tr_y, BACKBONES["vgg16"][1], train=True)))
print(x.shape, x.dtype, y.shape, y.dtype)     # (32, 224, 224, 3) float32 (32,) int32
print("etiquetas del batch:", y.numpy()[:10])

(32, 224, 224, 3) <dtype: 'float32'> (32,) <dtype: 'int32'>
etiquetas del batch: [0 5 1 8 3 8 3 9 0 8]


## 4. Route A: VGG16 (ImageNet) fine-tuned sobre espectrogramas

In [5]:
results = {}
results["vgg16"] = run_experiment("vgg16")

Epoch 1/15


/Users/annacasanovaspoirier/miniconda3/envs/sms/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


23/23 - 92s - 4s/step - accuracy: 0.1168 - loss: 7.7547 - val_accuracy: 0.1875 - val_loss: 3.1013
Epoch 2/15
23/23 - 101s - 4s/step - accuracy: 0.1641 - loss: 4.9669 - val_accuracy: 0.3125 - val_loss: 2.3745
Epoch 3/15
23/23 - 103s - 4s/step - accuracy: 0.2211 - loss: 4.1568 - val_accuracy: 0.3750 - val_loss: 1.9368
Epoch 4/15
23/23 - 115s - 5s/step - accuracy: 0.2754 - loss: 3.5812 - val_accuracy: 0.4500 - val_loss: 1.7782
Epoch 5/15
23/23 - 104s - 5s/step - accuracy: 0.3324 - loss: 3.0118 - val_accuracy: 0.4625 - val_loss: 1.7237
Epoch 6/15
23/23 - 114s - 5s/step - accuracy: 0.3561 - loss: 2.5882 - val_accuracy: 0.5000 - val_loss: 1.4681
Epoch 7/15
23/23 - 116s - 5s/step - accuracy: 0.3727 - loss: 2.4726 - val_accuracy: 0.5500 - val_loss: 1.5208
Epoch 8/15
23/23 - 113s - 5s/step - accuracy: 0.4172 - loss: 2.1639 - val_accuracy: 0.5500 - val_loss: 1.4051
Epoch 9/15
23/23 - 124s - 5s/step - accuracy: 0.4604 - loss: 1.8623 - val_accuracy: 0.5750 - val_loss: 1.3449
Epoch 10/15
23/23 - 12

In [ ]:
evaluate(results["vgg16"], "VGG16 fine-tuned")
plot_history(results["vgg16"], "VGG16")

In [ ]:
def summary_table(results):
    return pd.DataFrame([{
        "model": k, "val_acc": r["val_acc"], "test_acc": r["acc"],
        "train_time_s": r["train_time"], "infer_ms_per_song": r["infer_time"] * 1000,
        "params": r["params"]} for k, r in results.items()]).round(4)

# Guardar probabilidades para el ensemble / solapamiento de errores (preguntas 2 y 3)
for k, r in results.items():
    np.save(f"probs_{k}.npy", r["probs"])
np.save("test_labels.npy", te_y)

summary_table(results)

## 5. Más experimentos (descomenta lo que quieras probar)

Cada ejecución es una fila para la tabla del informe. Compara con `val_acc` para decidir; el test es solo para la configuración final.

In [ ]:
FT_CONFIGS = {
    "vgg16_head":    dict(unfreeze_last=0),     # solo cabeza (probe lineal)
    "vgg16_block5":  dict(unfreeze_last=4),     # bloque 5
    "vgg16_block45": dict(unfreeze_last=8),     # bloques 4 y 5
    "vgg16_full":    dict(unfreeze_last=999),   # toda la red
}
for k, cfg in FT_CONFIGS.items():
    if k not in results:                         # no repite lo ya hecho
        results[k] = run_experiment("vgg16", **cfg)
        np.save(f"probs_{k}.npy", results[k]["probs"])

summary_table(results)

In [ ]:
def n_last_from(name, prefix):
    """Nº de capas finales a descongelar para entrenar desde el bloque `prefix` en adelante."""
    Base, _ = BACKBONES[name]
    base = Base(include_top=False, weights=None, input_shape=IMG_SIZE + (3,))
    idx = next(i for i, l in enumerate(base.layers) if l.name.startswith(prefix))
    return len(base.layers) - idx

# Equivalencias que he comprobado:
#  vgg16:    block5 -> 4,  block4 -> 8
#  resnet50: conv5_block1 -> 32, conv4_block1 -> 94
#  effnetb0: block7a -> 16,  block6a -> 75

In [ ]:
CONFIGS = {
    # --- ResNet50 ---
    "resnet50_head":   ("resnet50", dict(unfreeze_last=0)),
    "resnet50_conv5":  ("resnet50", dict(unfreeze_last=n_last_from("resnet50", "conv5_block1"))),
    "resnet50_conv45": ("resnet50", dict(unfreeze_last=n_last_from("resnet50", "conv4_block1"))),
    "resnet50_full":   ("resnet50", dict(unfreeze_last=999)),
    # --- EfficientNetB0 ---
    "effnetb0_head":   ("effnetb0", dict(unfreeze_last=0)),
    "effnetb0_last":   ("effnetb0", dict(unfreeze_last=n_last_from("effnetb0", "block7a"))),
    "effnetb0_last2":  ("effnetb0", dict(unfreeze_last=n_last_from("effnetb0", "block6a"))),
    "effnetb0_full":   ("effnetb0", dict(unfreeze_last=999)),
}

for key, (backbone, cfg) in CONFIGS.items():
    if key in results:                       # no repite lo ya hecho
        continue
    results[key] = run_experiment(backbone, **cfg)
    np.save(f"probs_{key}.npy", results[key]["probs"])

summary_table(results)

In [ ]:
# results["resnet50"] = run_experiment("resnet50")
# results["effnetb0"] = run_experiment("effnetb0")

# Variantes de VGG16:
# results["vgg16_frozen"]  = run_experiment("vgg16", unfreeze_last=0)             # solo cabeza
# results["vgg16_ft10"]    = run_experiment("vgg16", unfreeze_last=10)
# results["vgg16_ft_lr1e4"] = run_experiment("vgg16", lr_ft=1e-4)
# results["vgg16_dense256"] = run_experiment("vgg16", dense_units=256)

summary_table(results)

## 6. (Opcional) Route B: VGGish, embeddings congelados + clasificador clásico

Necesita `pip install tensorflow_hub librosa` y los WAV en `genres_original`.
VGGish de TF Hub no es entrenable (solo extrae embeddings).

In [ ]:
# %pip install -q tensorflow_hub librosa
import tensorflow_hub as hub, librosa
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

vggish = hub.load("https://tfhub.dev/google/vggish/1")

def png_to_wav(p):
    """.../classical/classical00026.png -> genres_original/classical/classical.00026.wav"""
    p = Path(p); g = p.parent.name; num = p.stem[len(g):]
    return WAV_DIR / g / f"{g}.{num}.wav"

def embed(paths):
    out = []
    for p in paths:
        y, _ = librosa.load(png_to_wav(p), sr=16000, mono=True)
        e = vggish(y.astype(np.float32)).numpy()                  # (frames, 128)
        out.append(np.concatenate([e.mean(0), e.std(0)]))         # 256-d por canción
    return np.array(out)

t0 = time.time()
X_tr, X_va, X_te = embed(tr_p), embed(va_p), embed(te_p)
print("tiempo de embeddings:", round(time.time() - t0), "s")

clf = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000))
clf.fit(X_tr, tr_y)
print("val:", clf.score(X_va, va_y), "| test:", clf.score(X_te, te_y))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def extract_embeddings(name, paths):
    Base, prep = BACKBONES[name]
    base = Base(include_top=False, weights=WEIGHTS, input_shape=IMG_SIZE + (3,), pooling="avg")
    ds = make_ds(paths, np.zeros(len(paths)), prep)      # etiquetas ficticias
    return base.predict(ds, verbose=0)                   # (N, 512) en VGG16

t0 = time.time()
Xtr, Xva, Xte = [extract_embeddings("vgg16", p) for p in (tr_p, va_p, te_p)]
print("embeddings:", round(time.time() - t0), "s")

clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr, tr_y)
print("val:", clf.score(Xva, va_y), "| test:", clf.score(Xte, te_y))

## Código de VFFish

In [ ]:
# %pip install -q tensorflow_hub librosa
import tensorflow_hub as hub, librosa
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

vggish = hub.load("https://tfhub.dev/google/vggish/1")

def png_to_wav(p):
    """.../classical/classical00026.png -> genres_original/classical/classical.00026.wav"""
    p = Path(p); g = p.parent.name
    return WAV_DIR / g / f"{g}.{p.stem[len(g):]}.wav"

def embed_song(wav_path):
    y, _ = librosa.load(wav_path, sr=16000, mono=True)
    return vggish(y.astype(np.float32)).numpy()            # (n_frames, 128)

def embed_split(paths, cache_file):
    """Calcula (o carga de caché) un vector de 256 dimensiones por canción."""
    cache_file = Path(cache_file)
    if cache_file.exists():
        d = np.load(cache_file)
        assert len(d["X"]) == len(paths), f"{cache_file} no coincide con el split: bórralo"
        return d["X"], float(d["sec"])
    t0, X = time.time(), []
    for i, p in enumerate(paths):
        wav = png_to_wav(p)
        assert wav.exists(), f"No existe {wav}"
        fr = embed_song(wav)
        X.append(np.concatenate([fr.mean(0), fr.std(0)]))
        if i % 100 == 0: print(f"{cache_file.stem}: {i}/{len(paths)}")
    X, sec = np.array(X), time.time() - t0
    np.savez(cache_file, X=X, sec=sec)
    return X, sec

Xtr, s1 = embed_split(tr_p, "vggish_train.npz")
Xva, s2 = embed_split(va_p, "vggish_val.npz")
Xte, s3 = embed_split(te_p, "vggish_test.npz")
embed_ms = (s1 + s2 + s3) / (len(tr_p) + len(va_p) + len(te_p)) * 1000
print(f"embeddings: {s1+s2+s3:.0f} s en total | {embed_ms:.0f} ms/canción")

In [ ]:
def sk_result(name, clf, Xtr, Xva, Xte, embed_ms_per_song=0.0):
    t0 = time.time(); clf.fit(Xtr, tr_y); train_time = time.time() - t0
    t0 = time.time(); probs = clf.predict_proba(Xte); clf_ms = (time.time() - t0) / len(Xte) * 1000
    preds = probs.argmax(1)
    last = clf[-1] if hasattr(clf, "steps") else clf
    n_params = last.coef_.size + last.intercept_.size if hasattr(last, "coef_") else np.nan
    return dict(name=name, model=clf, probs=probs, preds=preds, acc=accuracy_score(te_y, preds),
                val_acc=clf.score(Xva, va_y), train_time=train_time,
                infer_time=(embed_ms_per_song + clf_ms) / 1000,   # incluye extraer el embedding
                params=n_params, hist=({}, {}))

SK_MODELS = {
    "vggish_logreg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "vggish_svm":    make_pipeline(StandardScaler(), SVC(probability=True, random_state=SEED)),
    "vggish_rf":     RandomForestClassifier(n_estimators=300, random_state=SEED),
}
for k, clf in SK_MODELS.items():
    results[k] = sk_result(k, clf, Xtr, Xva, Xte, embed_ms_per_song=embed_ms)
    np.save(f"probs_{k}.npy", results[k]["probs"])

summary_table(results)